# LangGraph Advanced: Memory, Human-in-the-Loop & Multi-Agent Supervisors

**Covers:**
1. `create_agent` — the modern prebuilt agent (successor to `langgraph.prebuilt.create_react_agent`)
2. Memory — multi-turn conversations via a checkpointer + `thread_id`
3. Human-in-the-loop — pausing before a sensitive tool call for approval, edit, or rejection
4. Multi-agent supervisor — an LLM router dispatching between specialist agents

In [3]:
!uv add langchain langgraph langchain-openai python-dotenv

Resolved 127 packages in 3ms
Checked 122 packages in 2ms


## Setup

Same pattern as the earlier notebooks: load the key from `.env`, build one `ChatOpenAI` instance shared across every section below.

In [4]:
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv

load_dotenv()

llm = ChatOpenAI(model="gpt-4o-mini")

## 1. `create_agent`: the modern prebuilt agent

`langgraph.ipynb` built the ReAct loop (LLM node + `ToolNode` + conditional edges + `add_messages` state) by hand. `langgraph.prebuilt.create_react_agent` used to be the one-call shortcut for that; it's now deprecated in favor of `langchain.agents.create_agent`, built on the same `StateGraph` machinery but with a **middleware** system — hooks that run before/after the model or a tool call. That middleware system is what Sections 3 and 4 build on.

In [5]:
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langchain.agents import create_agent

@tool
def calculate_sum(a: int, b: int) -> int:
    """Calculate the sum of two numbers."""
    return a + b

math_agent = create_agent(
    model=llm,
    tools=[calculate_sum],
    system_prompt="You are a helpful assistant. Use tools when they apply.",
)

In [6]:
result = math_agent.invoke({
    "messages": [HumanMessage(content="What is 12 + 30?")]
})

for message in result["messages"]:
    message.pretty_print()

================================ Human Message =================================

What is 12 + 30?
================================== Ai Message ==================================
Tool Calls:
  calculate_sum (call_bbiiAeyA4cxyOqIZr97GDiyy)
 Call ID: call_bbiiAeyA4cxyOqIZr97GDiyy
  Args:
    a: 12
    b: 30
================================= Tool Message =================================
Name: calculate_sum

42
================================== Ai Message ==================================

The sum of 12 and 30 is 42.


Same ReAct behavior as notebook 1 (tool call → tool result → final answer), zero `StateGraph`/`ToolNode`/`add_conditional_edges` boilerplate. `system_prompt` replaces the manual `SystemMessage` you'd otherwise prepend to `state["messages"]`.

## 2. Memory: multi-turn conversations via a checkpointer

`agent.invoke(...)` is stateless on its own — every call starts from whatever `messages` you pass in. A **checkpointer** persists graph state after every step, keyed by a `thread_id` in the call's `config`; passing the same `thread_id` again resumes from where that conversation left off, so you only ever send the *new* message. `InMemorySaver` is dev-only (state lives in process memory) — production deployments swap in `SqliteSaver` or `PostgresSaver` from the same `BaseCheckpointSaver` interface without touching agent code.

In [7]:
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()

agent_with_memory = create_agent(
    model=llm,
    tools=[calculate_sum],
    checkpointer=checkpointer,
)

config = {"configurable": {"thread_id": "session-1"}}

In [8]:
result = agent_with_memory.invoke(
    {"messages": [HumanMessage(content="My name is Surendra. What is 4 + 5?")]},
    config=config,
)
result["messages"][-1].pretty_print()

================================== Ai Message ==================================

Hello Surendra! The sum of 4 + 5 is 9.


In [7]:
result = agent_with_memory.invoke(
    {"messages": [HumanMessage(content="What's my name, and what did I just ask you to add?")]},
    config=config,
)
result["messages"][-1].pretty_print()

================================== Ai Message ==================================

Your name is Surendra, and you asked me to add 4 and 5.


The second call never repeated the name or the sum — the checkpointer replayed the full history for `thread_id="session-1"` before the model saw the new message. A different `thread_id` starts a blank conversation:

In [8]:
other_config = {"configurable": {"thread_id": "session-2"}}

result = agent_with_memory.invoke(
    {"messages": [HumanMessage(content="What's my name?")]},
    config=other_config,
)
result["messages"][-1].pretty_print()

================================== Ai Message ==================================

I'm sorry, but I don't know your name. Could you please tell me what it is?


## 3. Human-in-the-loop: approval gates on sensitive tools

Not every tool call should run unattended — paging someone, sending an email, deleting a resource. `HumanInTheLoopMiddleware` wraps specific tools: when the model requests one of them, the graph **interrupts** (pauses and persists state via the checkpointer) instead of executing it. `agent.invoke(...)` returns with an `__interrupt__` key describing the pending action; resuming with a `Command(resume=...)` carrying a decision (`approve`, `edit`, `reject`, or `respond`) continues the graph from exactly where it paused.

In [9]:
@tool
def send_alert(message: str) -> str:
    """Page the on-call engineer with an alert message."""
    return f"Paged on-call: {message}"

In [10]:
from langchain.agents.middleware import HumanInTheLoopMiddleware

oncall_agent = create_agent(
    model=llm,
    tools=[send_alert],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={"send_alert": {"allowed_decisions": ["approve", "reject"]}}
        )
    ],
    checkpointer=InMemorySaver(),
)

hitl_config = {"configurable": {"thread_id": "hitl-1"}}

result = oncall_agent.invoke(
    {"messages": [HumanMessage(content="Page on-call about the prod database outage")]},
    config=hitl_config,
)

result["__interrupt__"]

[Interrupt(value={'action_requests': [{'name': 'send_alert', 'args': {'message': 'URGENT: Production database outage detected. Immediate attention required.'}, 'description': "Tool execution requires approval\n\nTool: send_alert\nArgs: {'message': 'URGENT: Production database outage detected. Immediate attention required.'}"}], 'review_configs': [{'action_name': 'send_alert', 'allowed_decisions': ['approve', 'reject']}]}, id='36a862ce5774eecb36bb24696428c7ab')]

The graph stopped *before* running `send_alert` and surfaced the pending call for review, instead of paging anyone. Resuming with an `approve` decision lets it through:

In [12]:
from langgraph.types import Command

result = oncall_agent.invoke(
    Command(resume={"decisions": [{"type": "approve"}]}),
    config=hitl_config,
)

for message in result["messages"]:
    message.pretty_print()

================================ Human Message =================================

Page on-call about the prod database outage
================================== Ai Message ==================================
Tool Calls:
  send_alert (call_fN1g4zL9mSHTSJBwhxBc0iz9)
 Call ID: call_fN1g4zL9mSHTSJBwhxBc0iz9
  Args:
    message: URGENT: Production database outage detected. Immediate attention required.
================================= Tool Message =================================
Name: send_alert

Paged on-call: URGENT: Production database outage detected. Immediate attention required.
================================== Ai Message ==================================

The on-call engineer has been paged about the production database outage with the alert message.


A `reject` decision (on a fresh thread — each thread's interrupt can only be resolved once) skips execution and tells the model why, so it can respond accordingly instead of retrying:

In [13]:
reject_config = {"configurable": {"thread_id": "hitl-2"}}

oncall_agent.invoke(
    {"messages": [HumanMessage(content="Page on-call about a minor disk space warning")]},
    config=reject_config,
)

result = oncall_agent.invoke(
    Command(resume={"decisions": [{
        "type": "reject",
        "message": "Not urgent enough to page - log it instead",
    }]}),
    config=reject_config,
)

for message in result["messages"]:
    message.pretty_print()

================================ Human Message =================================

Page on-call about a minor disk space warning
================================== Ai Message ==================================
Tool Calls:
  send_alert (call_yK42iZjXjs4D8KzgIPzZwkbD)
 Call ID: call_yK42iZjXjs4D8KzgIPzZwkbD
  Args:
    message: Minor disk space warning has been detected. Please investigate.
================================= Tool Message =================================
Name: send_alert

User rejected the tool call for `send_alert` with reason: Not urgent enough to page - log it instead
================================== Ai Message ==================================

It seems that the on-call engineer prefers to log the minor disk space warning instead of being paged. If you have a logging system in place, please log the warning accordingly. If you need assistance with that, let me know!


## 4. Multi-agent supervisor: routing between specialists

A single agent with every tool bound gets unwieldy — tool descriptions collide, the model second-guesses which one applies (recall the sum-vs-multiply tool question from the `langgraph.ipynb` walkthrough). The common fix is a **supervisor**: a small router that classifies the request and dispatches to a specialist agent built — and prompted — for just that job. Each specialist here is itself a `create_agent` graph; the supervisor is a `StateGraph` wired by hand, same pattern as notebook 1, but its nodes are compiled agents instead of single LLM calls.

In [14]:
@tool
def calculate_product(a: int, b: int) -> int:
    """Calculate the product of two numbers."""
    return a * b

RUNBOOKS = {
    "crashloop": "Check `kubectl describe pod` for the exit code, verify readiness/liveness probes, and check resource limits.",
    "node not ready": "Check kubelet logs, node conditions via `kubectl describe node`, and disk pressure.",
}

@tool
def lookup_runbook(topic: str) -> str:
    """Look up a platform/SRE runbook snippet by topic (e.g. 'crashloop', 'node not ready')."""
    return RUNBOOKS.get(topic.lower(), "No runbook found for this topic.")

math_specialist = create_agent(
    model=llm,
    tools=[calculate_sum, calculate_product],
    system_prompt="You are a math specialist. Use tools for arithmetic.",
)

docs_specialist = create_agent(
    model=llm,
    tools=[lookup_runbook],
    system_prompt="You are a platform/SRE runbook specialist. Use the lookup_runbook tool.",
)

**The router**: `with_structured_output` (same mechanism as the OpenAI notebook's structured-output section) forces the classification into a fixed `Literal`, instead of trusting the model to return an exact string.

In [15]:
from typing import TypedDict, Annotated, Literal
from pydantic import BaseModel
from langchain_core.messages import SystemMessage
from langgraph.graph.message import add_messages

class Route(BaseModel):
    next: Literal["math", "docs"]

router = llm.with_structured_output(Route)

class SupervisorState(TypedDict):
    messages: Annotated[list, add_messages]
    next: str

def supervisor(state: SupervisorState):
    decision = router.invoke([
        SystemMessage(content="Classify the user's request. Route to 'math' for arithmetic questions, 'docs' for platform/SRE runbook lookups."),
        state["messages"][-1],
    ])
    return {"next": decision.next}

Each specialist node hands the full message history to its own `create_agent` graph and returns its updated messages — `add_messages` matches on message `id`, so messages already in state are merged rather than duplicated.

In [16]:
def call_math_specialist(state: SupervisorState):
    result = math_specialist.invoke({"messages": state["messages"]})
    return {"messages": result["messages"]}

def call_docs_specialist(state: SupervisorState):
    result = docs_specialist.invoke({"messages": state["messages"]})
    return {"messages": result["messages"]}

In [17]:
from langgraph.graph import StateGraph, START, END

graph = StateGraph(SupervisorState)

graph.add_node("supervisor", supervisor)
graph.add_node("math", call_math_specialist)
graph.add_node("docs", call_docs_specialist)

graph.add_edge(START, "supervisor")

graph.add_conditional_edges(
    "supervisor",
    lambda state: state["next"],
    {"math": "math", "docs": "docs"},
)

graph.add_edge("math", END)
graph.add_edge("docs", END)

supervisor_app = graph.compile()

In [18]:
result = supervisor_app.invoke({
    "messages": [HumanMessage(content="What is 8 times 9?")]
})
for message in result["messages"]:
    message.pretty_print()

================================ Human Message =================================

What is 8 times 9?
================================== Ai Message ==================================
Tool Calls:
  calculate_product (call_O7RKeJNFwT7xnKjr9eHUPF5E)
 Call ID: call_O7RKeJNFwT7xnKjr9eHUPF5E
  Args:
    a: 8
    b: 9
================================= Tool Message =================================
Name: calculate_product

72
================================== Ai Message ==================================

8 times 9 is 72.


In [19]:
result = supervisor_app.invoke({
    "messages": [HumanMessage(content="One of my pods keeps crashlooping, what should I check?")]
})
for message in result["messages"]:
    message.pretty_print()

================================ Human Message =================================

One of my pods keeps crashlooping, what should I check?
================================== Ai Message ==================================
Tool Calls:
  lookup_runbook (call_XSV9pzxOgP1W8tkXAxDRivns)
 Call ID: call_XSV9pzxOgP1W8tkXAxDRivns
  Args:
    topic: crashloop
================================= Tool Message =================================
Name: lookup_runbook

Check `kubectl describe pod` for the exit code, verify readiness/liveness probes, and check resource limits.
================================== Ai Message ==================================

When a pod is crashlooping, you should check the following:

1. Run `kubectl describe pod <pod-name>` to check the exit code of the container. This can provide insight into why the pod is crashing.
2. Verify the readiness and liveness probes configured for the pod. Incorrect configurations can cause the pod to fail health checks and restart.
3. Check the 

## Recap

- **`create_agent`** (successor to `langgraph.prebuilt.create_react_agent`) collapses the hand-rolled ReAct graph from notebook 1 into one call, and adds a **middleware** hook system
- **Memory**: a `checkpointer` + `thread_id` persists state across `.invoke()` calls — swap `InMemorySaver` for `SqliteSaver`/`PostgresSaver` in production without touching agent code
- **Human-in-the-loop**: `HumanInTheLoopMiddleware` interrupts before named tools run; the graph pauses (state saved via the checkpointer) and resumes via `Command(resume={"decisions": [...]})` with `approve` / `edit` / `reject` / `respond`
- **Multi-agent supervisor**: a router (here, structured-output classification) dispatches to specialist `create_agent` graphs, each with a narrow, unambiguous toolset instead of one agent juggling every tool
- It all composes: a supervisor's specialists could just as easily carry their own checkpointer or HITL middleware